In [90]:
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import pacf

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor


from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Read file

In [91]:



file_path = './dataset/observations_part_001.parquet'
df_raw = pd.read_parquet(file_path)

print(f"Row Total: {df_raw.shape[0]:,}")
print(f"Column Total: {df_raw.shape[1]}")

Row Total: 480,000
Column Total: 59


# Data Info


In [92]:
df_raw.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480000 entries, 0 to 479999
Data columns (total 59 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   activity_multiplier           480000 non-null  float32
 1   animal_residue_pct            480000 non-null  int32  
 2   battery_level_percent         477785 non-null  float32
 3   battery_maintenance_event     480000 non-null  int32  
 4   bin_index                     480000 non-null  int32  
 5   collection_delay_minutes      480000 non-null  float32
 6   collection_event              480000 non-null  int32  
 7   collection_threshold_percent  480000 non-null  float32
 8   college_calendar_closed       480000 non-null  int32  
 9   date_index                    480000 non-null  int32  
 10  day_of_week                   480000 non-null  int32  
 11  dominant_waste_code           480000 non-null  int32  
 12  event_code                    480000 non-nul

In [93]:
df_raw.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,sensor_error_code,temperature_c,textile_pct,time_slot_code,timestamp_unix,true_fill_level_percent,waste_bulk_density_kg_m3,waste_entering_bin_kg,waste_volume_liter,weather_code
0,0.6984,4,96.482681,0,0,0.0,0,80.0,0,4,...,0,13.053044,6,0,1736013600,49.544697,226.006271,0.629307,2.784469,0
1,0.5626,0,92.744682,0,1,0.0,0,80.0,0,4,...,0,14.072392,1,0,1736013600,49.981346,251.556946,0.622490,2.474549,0
2,0.8924,1,96.756447,0,2,0.0,0,80.0,0,4,...,0,13.722422,4,0,1736013600,32.794777,232.091263,1.656849,7.138784,0
3,0.2134,3,95.200546,0,3,0.0,0,75.0,0,4,...,0,13.480151,4,0,1736013600,24.558718,199.116287,0.294287,1.477966,0
4,0.8924,0,92.272179,0,4,0.0,0,75.0,0,4,...,0,13.799287,3,0,1736013600,67.464378,222.365173,2.896791,13.027180,0


# Cleaning Data
Ambil 50 tempat sampah dan urutkan


In [94]:
df_50 = df_raw[df_raw['bin_index'] < 50].sort_values(['bin_index', 'timestamp_unix']).reset_index(drop=True)

## Anomaly Handing
Hapus nilai jika sensor error/packet loss, lalu lakukan interpolasi (pengisian nilai tengah)

In [95]:
anomaly_mask = (df_50['is_sensor_anomaly'] == 1) | (df_50['is_packet_loss'] == 1)
print(f"-> Ditemukan {anomaly_mask.sum()} baris anomali/error hardware. Melakukan Interpolasi...")
df_50.loc[anomaly_mask, 'fill_level_percent'] = np.nan
df_50['fill_level_percent'] = df_50.groupby('bin_index')['fill_level_percent'].transform(
    lambda x: x.interpolate(method='linear').bfill().ffill()
)

-> Ditemukan 28 baris anomali/error hardware. Melakukan Interpolasi...


### Statistical Justification
Uji PACF untuk menjustifikasi kenapa pakai riwayat 3 Jam


In [96]:
pacf_vals = pacf(df_50[df_50['bin_index'] == 0]['fill_level_percent'].dropna(), nlags=5)
print(f"Uji PACF (Partial Autocorrelation) pada Bin 0:")
for i, val in enumerate(pacf_vals):
    print(f"  Lag {i} jam lalu: {val:.3f} (Jika > 0.1, artinya signifikan secara statistik)")




Uji PACF (Partial Autocorrelation) pada Bin 0:
  Lag 0 jam lalu: 1.000 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 1 jam lalu: 0.927 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 2 jam lalu: -0.021 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 3 jam lalu: -0.040 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 4 jam lalu: -0.053 (Jika > 0.1, artinya signifikan secara statistik)
  Lag 5 jam lalu: -0.020 (Jika > 0.1, artinya signifikan secara statistik)


In [97]:
df_proc = df_50.copy()

#### A. Cyclical Time Encoding (Mencegah AI bingung perpindahan Jam 23 ke Jam 0)

In [98]:
df_proc['hour_sin'] = np.sin(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['hour_cos'] = np.cos(2 * np.pi * df_proc['hour'] / 24.0)
df_proc['day_sin'] = np.sin(2 * np.pi * df_proc['day_of_week'] / 7.0)
df_proc['day_cos'] = np.cos(2 * np.pi * df_proc['day_of_week'] / 7.0)

#### B. Lag Features


In [99]:
df_proc['lag_1h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(1)
df_proc['lag_2h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(2)
df_proc['lag_3h'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(3)

#### C. Ekstraksi Velocity (Laju percepatan sampah masuk)


In [100]:
df_proc['velocity_1h'] = df_proc['fill_level_percent'] - df_proc['lag_1h']
df_proc['velocity_2h'] = df_proc['lag_1h'] - df_proc['lag_2h']

#### D. Pembuatan Target Output (Kapasitas 1 Jam ke depan)


In [101]:
df_proc['target_next_hour'] = df_proc.groupby('bin_index')['fill_level_percent'].shift(-1)



##### Buang NaN


In [102]:
df_proc = df_proc.dropna().reset_index(drop=True)

### Feature Selection

In [103]:
features_engineered = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]

Mengambil sampel 5000 baris agar komputasi MI cepat


In [104]:
n_sample = min(5000, len(df_proc))
sampel_mi = df_proc.sample(n_sample, random_state=42)

mi_scores = mutual_info_regression(sampel_mi[features_engineered], sampel_mi['target_next_hour'], random_state=42)
mi_df = pd.DataFrame({'Fitur': features_engineered, 'Skor_MI': mi_scores}).sort_values('Skor_MI', ascending=False)
print("Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):")
print(mi_df.to_string(index=False))

Information Gain tiap fitur terhadap Target (Nilai lebih tinggi = Lebih penting):
             Fitur  Skor_MI
fill_level_percent 2.018102
            lag_1h 1.794074
            lag_2h 1.591467
            lag_3h 1.434640
       velocity_1h 0.033735
       velocity_2h 0.028473
           day_cos 0.000136
          hour_sin 0.000000
          hour_cos 0.000000
           day_sin 0.000000


## Train-Test Split


In [105]:
train_mask = df_proc['bin_index'] < 40  # 40 Bin untuk Belajar
test_mask = df_proc['bin_index'] >= 40  # 10 Bin untuk Ujian

y_train = df_proc.loc[train_mask, 'target_next_hour']
y_test = df_proc.loc[test_mask, 'target_next_hour']

## Experimental A (Using Raw Data)

In [106]:
basic_cols = ['hour', 'day_of_week', 'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h']
rf_basic = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_basic.fit(df_proc.loc[train_mask, basic_cols], y_train)
pred_basic = rf_basic.predict(df_proc.loc[test_mask, basic_cols])
mae_basic = mean_absolute_error(y_test, pred_basic)

## Experimental B (Using Feature Engineering)

In [107]:
rf_featured = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_featured.fit(df_proc.loc[train_mask, features_engineered], y_train)
pred_rf = rf_featured.predict(df_proc.loc[test_mask, features_engineered])
mae_rf = mean_absolute_error(y_test, pred_rf)

## Experimental C (Comprehensive Model & Scaler Comparison)
Menguji Linear Regression, Random Forest, dan XGBoost terhadap berbagai jenis Feature Scaling.


In [108]:
# 1. Dictionary of Scalers
scalers_dict = {
    'Tanpa Scaler': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# 2. Dictionary of Models
models_dict = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
}

# 3. Running Grid Comparison
grid_results = []

print("Mulai melatih 12 kombinasi model... Mohon tunggu.")

for scaler_name, scaler_obj in scalers_dict.items():
    # Proses Scaling Data
    if scaler_obj is None:
        X_tr = df_proc.loc[train_mask, features_engineered]
        X_te = df_proc.loc[test_mask, features_engineered]
    else:
        # Fit & Transform pada data latih
        X_tr = scaler_obj.fit_transform(df_proc.loc[train_mask, features_engineered])
        # Hanya Transform pada data uji (Mencegah data leakage)
        X_te = scaler_obj.transform(df_proc.loc[test_mask, features_engineered])
    
    # Menyiapkan baris hasil untuk tabel
    row_result = {'Scaler': scaler_name}
    
    # Melatih dan menguji tiap model
    for model_name, model_obj in models_dict.items():
        model_obj.fit(X_tr, y_train)
        pred = model_obj.predict(X_te)
        mae_score = mean_absolute_error(y_test, pred)
        row_result[model_name] = f"{mae_score:.3f}%"
        
    grid_results.append(row_result)

print("Selesai!\n")

Mulai melatih 12 kombinasi model... Mohon tunggu.
Selesai!



# Evaluation Result

In [109]:
# Tampilkan Hasil Komparasi Awal
print("=== PERBANDINGAN FITUR ===")
print(f"Model Rekan Anda (Waktu mentah, Tanpa Velocity) -> MAE: {mae_basic:.3f}%")
print(f"Model Anda (Random Forest + Feature Engineering) -> MAE: {mae_rf:.3f}%\n")

# Tampilkan Tabel Grid Hasil Scaler & Model Lanjutan
print("=== KOMPARASI LANJUTAN (MODEL vs SCALER) ===")
df_grid_results = pd.DataFrame(grid_results)
print(df_grid_results.to_string(index=False))

=== PERBANDINGAN FITUR ===
Model Rekan Anda (Waktu mentah, Tanpa Velocity) -> MAE: 3.301%
Model Anda (Random Forest + Feature Engineering) -> MAE: 3.146%

=== KOMPARASI LANJUTAN (MODEL vs SCALER) ===
        Scaler Linear Regression Random Forest XGBoost
  Tanpa Scaler            3.198%        3.146%  3.178%
StandardScaler            3.198%        3.158%  3.181%
  MinMaxScaler            3.198%        3.161%  3.180%
  RobustScaler            3.198%        3.169%  3.204%


# Save Dataset

In [88]:
df_proc.to_csv('dataset_trainable.csv', index=False)